# 7. 模型保存与加载

训练好的模型需要保存下来，下次直接加载使用，不用重新训练。

### 两种保存方式

1. **保存参数（推荐）**：`torch.save(model.state_dict(), 'model.pth')`，只保存模型的权重参数，文件小、跨环境兼容
2. **保存整个模型（不推荐）**：`torch.save(model, 'model.pth')`，保存整个对象，跨环境容易出问题

### 为什么推荐保存参数？

保存整个模型会依赖代码的目录结构，换台电脑可能就加载不了。只保存参数则不受限制，只要模型定义一样就行。

In [1]:
import torch
import torch.nn as nn

In [2]:
# 定义一个简单的模型
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(20, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

model = SimpleNet()
print("模型结构:")
print(model)

模型结构:
SimpleNet(
  (fc1): Linear(in_features=20, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=10, bias=True)
)


## 7.1 保存模型参数

In [3]:
# 保存模型参数
torch.save(model.state_dict(), 'model.pth')
print("模型参数已保存到 model.pth")

# state_dict() 返回一个字典：{参数名: 参数值}
print("\n参数字典的 key:")
for key, value in model.state_dict().items():
    print(f"  {key}: {value.shape}")

模型参数已保存到 model.pth

参数字典的 key:
  fc1.weight: torch.Size([64, 20])
  fc1.bias: torch.Size([64])
  fc2.weight: torch.Size([10, 64])
  fc2.bias: torch.Size([10])


## 7.2 加载模型参数

加载时需要先创建一个**相同结构的模型**，然后把参数填进去。

In [4]:
# 加载模型参数
new_model = SimpleNet()  # 先创建同结构的空模型
new_model.load_state_dict(torch.load('model.pth'))  # 填入保存的参数
new_model.eval()  # 切换到评估模式

print("模型参数已加载")

# 验证：两次前向传播结果完全一致
x = torch.randn(1, 20)
original_output = model(x)
loaded_output = new_model(x)
print(f"原始模型输出: {original_output}")
print(f"加载模型输出: {loaded_output}")
print(f"结果一致: {torch.allclose(original_output, loaded_output)}")

模型参数已加载
原始模型输出: tensor([[-0.0055, -0.5644, -0.1942,  0.0098,  0.3239,  0.1134, -0.1903,  0.4265,
          0.0668,  0.0115]], grad_fn=<AddmmBackward0>)
加载模型输出: tensor([[-0.0055, -0.5644, -0.1942,  0.0098,  0.3239,  0.1134, -0.1903,  0.4265,
          0.0668,  0.0115]], grad_fn=<AddmmBackward0>)
结果一致: True


## 7.3 实际训练中的保存

实际训练时，通常把优化器状态、epoch、损失等也一起保存，方便恢复训练：

```python
# 保存检查点
checkpoint = {
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss,
}
torch.save(checkpoint, 'checkpoint.pth')

# 恢复训练
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
```

In [5]:
# 清理保存的文件
import os
os.remove('model.pth')
print("已清理 model.pth")

已清理 model.pth
